# Notebook 12 -- export bundle

**Asks: write everything the R Markdown reads, plus the archive bundle.**

This is the last notebook in the chain, and it runs no analysis of its own. It
does three things.

1. **Reports the state of the chain.** It reads the manifest -- the small file
   every notebook writes into when it finishes -- and says which notebooks ran,
   in what mode, when, and which outputs are not there. If you have been away
   from this project for a month, this printout is where you find out where you
   are.
2. **Writes the six files `02_nlp_integration.Rmd` reads**, under exactly the
   names that document looks for. Three of them are data this chain already
   holds under a different name. The R Markdown was written first, so its names
   are the ones that have to be matched.
3. **Zips the deposit.** Prereg 6.7 commits the project to posting the model set
   and versions, the scoring prompts, the word lists, the raw model output and
   the analysis code on the project's OSF page. The bundle is that deposit, and
   it travels with a written list of what it should contain, so an incomplete
   deposit is visible rather than silent.

**In** -- every output of the chain, discovered from the manifest. No input is
required: a file that is not there is reported, never demanded.
**Out** -- `for_rmd/` (the six R Markdown inputs), `osf_bundle_manifest.json`,
and `retribution_nlp_bundle.zip`.

### The one thing this notebook refuses to do

It will not deposit a file that carries the same response twice. Everything it
reads is optional and a missing input costs one printed line, but a file going
out to R with a repeated `response_id x item_type` key gives one participant
double weight in every correlation while the printed sample size stays the same,
and the R Markdown averages the repeat rather than objecting. That check stops
the export, names the file and names the notebook that has to be fixed.

### What this notebook cannot tell you

It cannot tell you a result. The gate verdict it prints is read back from
`gate_results.csv` exactly as the gate notebook left it, and where a clause did
not run it says "not run" rather than leaving the cell blank. A blank in a gate
column reads as a failure to anyone skimming the table, and a clause that was
never evaluated is not a clause that failed.

## Config

In [ ]:
# === CONFIG: the only cell you edit ===
# Every switch in the pipeline lives here. Nothing below this cell needs changing.

RUN_MODE    = "full"        # "smoke" = first 20 rows, costs pennies, proves the
                            #           whole chain works end to end.
                            # "full"  = the real corpus.
HUMAN_GOLD  = False         # False = the pre-registered pipeline. Language models
                            #         do the coding, exactly as prereg 6.7 says.
                            #         This is the default and it runs start to
                            #         finish with no human step.
                            # True  = ALSO run the human-coder arm. That arm is a
                            #         DEVIATION from the registration and needs the
                            #         OSF addendum posted first.
WORKDIR     = "/content/retribution_nlp"   # where every file in the chain lives
RANDOM_SEED = 20260629      # frozen. Changing it re-draws every random split.

# --- PER-NOTEBOOK: the three lines below differ in each notebook ---
NB_ID       = "12"                     # this notebook's number
NB_NAME     = "export_bundle"          # this notebook's job, in two words
NB_ASKS     = "Write everything the R Markdown reads, plus the archive bundle."

## Setup

In [ ]:
# === SETUP: folders, the manifest, and the two functions every notebook uses ===
# WHAT THIS CELL DOES: makes the working folder, loads the manifest, and defines
# need() and done(). You never edit this cell.
#
# THE MANIFEST is a small file called _manifest.json that lives beside the data.
# Every notebook records what it produced in there. The next notebook reads it to
# check its inputs exist. That is what lets a notebook tell you "run 03 first"
# instead of dying on a missing file three cells later.

import os, json, hashlib, datetime, sys

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
MANIFEST = os.path.join(WORKDIR, "_manifest.json")

def _load_manifest():
    """Read the manifest, or start an empty one on the first notebook."""
    if os.path.exists(MANIFEST):
        with open(MANIFEST) as f:
            return json.load(f)
    return {"files": {}, "runs": []}

def _sha256(path, cap=8_000_000):
    """First 8 MB is plenty to catch a file that changed underneath us."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(cap))
    return h.hexdigest()[:16]

class MissingInput(Exception):
    """Raised when a required upstream file is not there yet."""
    pass

def need(filename, produced_by=None, optional=False):
    """Return the path to an input file, or explain which notebook makes it.

    This is the function that makes the chain navigable. A missing input stops
    the notebook HERE, with a sentence telling you what to run, rather than
    surfacing as a confusing error further down.
    """
    path = os.path.join(WORKDIR, filename)
    if os.path.exists(path):
        rec = _load_manifest()["files"].get(filename, {})
        stamp = rec.get("written_at", "unknown time")
        print(f"  [ok]   {filename}  (written {stamp})")
        return path
    if optional:
        print(f"  [skip] {filename} is not here. This notebook will carry on "
              f"without it and say so in its output.")
        return None
    where = f" Run notebook {produced_by} first." if produced_by else ""
    raise MissingInput(
        f"\n\n  MISSING INPUT: {filename}\n"
        f"  This notebook cannot run without it.{where}\n"
        f"  Files present in {WORKDIR}:\n    "
        + "\n    ".join(sorted(os.listdir(WORKDIR)) or ["(none)"]) + "\n")

def done(filename, rows=None, note=""):
    """Record a file this notebook produced, so later notebooks can find it."""
    path = os.path.join(WORKDIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"done() called for {filename} but it is not there")
    m = _load_manifest()
    m["files"][filename] = {
        "written_by": f"{NB_ID}_{NB_NAME}",
        "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "rows": rows,
        "sha16": _sha256(path),
        "run_mode": RUN_MODE,
        "note": note,
    }
    m["runs"] = [r for r in m["runs"] if r.get("nb") != NB_ID]
    m["runs"].append({"nb": NB_ID, "name": NB_NAME, "run_mode": RUN_MODE,
                      "human_gold": HUMAN_GOLD,
                      "at": datetime.datetime.now().isoformat(timespec="seconds")})
    with open(MANIFEST, "w") as f:
        json.dump(m, f, indent=1)
    size = os.path.getsize(path)
    print(f"  [wrote] {filename}  ({rows if rows is not None else '?'} rows, "
          f"{size:,} bytes)")

print(f"NOTEBOOK {NB_ID} -- {NB_NAME}")
print(f"  {NB_ASKS}")
print(f"  mode: RUN_MODE={RUN_MODE}, HUMAN_GOLD={HUMAN_GOLD}")
print(f"  folder: {WORKDIR}")
_m = _load_manifest()
if _m["runs"]:
    print("  already run in this folder: "
          + ", ".join(sorted(r["nb"] for r in _m["runs"])))
else:
    print("  this folder is empty: notebook 00 is the place to start.")

## Load

In [ ]:
# === LOAD (1 of 3): the notebooks, by number ===
# WHAT THIS CELL DOES: names every notebook in the chain and what it is for.
#
# WHY IT IS WRITTEN DOWN HERE rather than read from somewhere. The manifest
# records the notebooks that HAVE run. A notebook that never ran leaves no trace
# in it at all, so a report built only from the manifest can list what happened
# and nothing else -- which is precisely the report you do not need when
# something did not happen. The list below is what lets the state report say
# "notebook 09 never ran" instead of only "a file is missing".
NB_JOB = {
    "00": "extract the open-ends from the Qualtrics export",
    "01": "score every answer with the nine-facet word list",
    "02": "ask which facets the word list can speak about at all",
    "03": "draw the model-free sample, freeze the SEED/TEST split",
    "04": "build the embedding reference pools",
    "05": "write the one codebook, and smoke-test every model leg",
    "06": "score the corpus with ONE language model (once per model)",
    "07": "merge the models, gate clause B (inter-model ICC)",
    "08": "embed the corpus with ONE encoder (once per encoder)",
    "09": "facet similarity from the embeddings",
    "10": "the gate: clause C, and the union of the clauses",
    "11": "the E3 to E7 analyses and the one FDR family",
    "12": "this notebook: the R Markdown inputs and the bundle",
    "99": "the optional human-coder arm (HUMAN_GOLD only)",
}

# NOTEBOOK 99 IS THE ONE OPTIONAL NOTEBOOK IN THE CHAIN. Prereg 6.7 has an
# ensemble of language models code the corpus "in place of human raters" and
# prescribes no human-coding step, so with HUMAN_GOLD False the chain runs start
# to finish without 99, and gate clause A -- agreement with human coders, which is
# our addition rather than a registered requirement -- is reported as not run.
# Its absence is the registered pipeline, not a hole in it.
#
# NOTEBOOK 05 IS NOT THE OPTIONAL ONE. It writes rubric.json, the single codebook,
# and notebook 06 opens with need("rubric.json", produced_by="05"): nothing is
# scored until 05 has run, whatever HUMAN_GOLD says.
OPTIONAL_NB = "99"     # read by the state report, by the expected-output table and
                       # by the export below, so "optional" is decided in one place
print(f"The chain is {len(NB_JOB) - 1} notebooks, 00 through 12, plus notebook "
      f"{OPTIONAL_NB}, the optional human-coder arm.")
print(f"HUMAN_GOLD is {HUMAN_GOLD}: notebook {OPTIONAL_NB} "
      + ("should have run, and clause A is expected."
         if HUMAN_GOLD else
         "is not part of this run, and clause A will read 'not run'."))

In [ ]:
# === LOAD (2 of 3): what each notebook is supposed to have left behind ===
# WHAT THIS CELL DOES: lists every file the chain produces, with the notebook that
# produces it and what its absence costs. Nothing is read yet.
#
# WHY A TABLE AS WELL AS need() CALLS. need() is how a notebook demands an input,
# and this notebook uses it too -- in the export cell below, with optional=True,
# which prints one line for a missing file and carries on. But need() can only
# speak about a file the notebook goes looking for, and this one has to report on
# files it never opens: the raw model output, the embedding caches, the clause-C
# matrix. Those belong to the deposit and to nothing else, so they are listed here
# or they drop out of the report in silence.
#
# The last field says what a missing file costs in terms of the analysis it
# blocks. A list of missing names is only useful if the reader can see the price.
#
#   core   something downstream cannot be done without it.
#   extra  the deposit is poorer without it, and no analysis stops.
#   human  the optional arm's output. Expected only when HUMAN_GOLD is True.
#   many   the name carries a model or encoder key and cannot be written down in
#          advance, so those rows are matched by prefix rather than by exact name.
CHAIN = [
    ("00", "openends_long.csv", "core",
     "the corpus itself. Every later notebook reads it."),
    ("00", "retribution_scored.csv", "core",
     "the closed-ended survey scales. Gate clause C is measured against them."),
    ("01", "facet_dictionary_scores.csv", "core",
     "the word-list stream, and one of the six files the R Markdown reads."),
    ("02", "word_list_coverage.csv", "extra",
     "which facets the word list fires on often enough to corroborate anything."),
    ("03", "strata_sample.csv", "extra",
     "the model-free sample the optional human arm would code."),
    ("03", "gold_split_assignment.csv", "core",
     "the frozen SEED/TEST split. The R Markdown reads it under this name."),
    ("04", "embedding_seed_pools.json", "core",
     "the reference passages every embedding similarity is measured against."),
    ("05", "rubric.json", "core",
     "the one codebook. Notebook 06 scores nothing at all without it."),
    ("05", "smoke_test_results.csv", "extra",
     "the cheap proof that every model leg parses, taken before the paid run."),
    ("06", "llm_scores_", "many",
     "two files per model: every call verbatim, and the same thing as a table."),
]

In [ ]:
# === LOAD (2 of 3, cont.): the rest of the expected-output table ===
CHAIN += [
    ("07", "ensemble_scores.csv", "core",
     "the facet score of record. Goes to R as ensemble_facet_scores.csv."),
    ("07", "inter_model_icc.csv", "core",
     "gate clause B. Goes to R as ensemble_agreement.csv."),
    ("08", "embeddings_", "many",
     "one embedding cache per encoder. Expensive to rebuild, so they travel."),
    ("09", "facet_similarity.csv", "core",
     "embedding similarity. Goes to R as embedding_facet_similarity.csv."),
    ("09", "embedding_model_sensitivity.csv", "extra",
     "whether the encoders order the responses the same way."),
    ("10", "gate_results.csv", "core",
     "the gate: which facets are gate-backed and which are descriptive-only."),
    ("10", "stream_agreement_matrix.csv", "core",
     "the three streams against each other: is a facet measured, or asserted."),
    ("10", "gate_c_facet_by_scale_matrix.csv", "core",
     "the whole clause-C matrix: each facet against its own scale and its rivals."),
    ("11", "E3_to_E7_results.csv", "core",
     "the E3 to E7 tables, stacked, with their block labels. The results table."),
    ("11", "fdr_exploratory_family_E3toE7.csv", "core",
     "the one corrected family -- the p_bh column that E3 to E7 share."),
    ("99", "gold_codes.csv", "human",
     "the adjudicated human codes. Exists only when HUMAN_GOLD is True."),
    ("99", "human_match_results.csv", "human",
     "gate clause A, our addition to the registered gate. HUMAN_GOLD only."),
    ("99", "gold_irr_by_facet.csv", "human",
     "how far the coders agreed with each other. HUMAN_GOLD only."),
]

# Derived here and read in four places below, so that "which files belong to the
# optional arm" is answered by the table above and never typed out a second time.
HUMAN_FILES = {name for nb, name, kind, cost in CHAIN if kind == "human"}

# THE FIVE THE DEPOSIT CANNOT GO UP WITHOUT. A gap in any one of them means the
# cell that writes it did not complete, most often because the run was partial.
# Between them they hold every number a reader would quote.
DEPOSIT_MUST_CARRY = ["gate_results.csv", "E3_to_E7_results.csv",
                      "stream_agreement_matrix.csv",
                      "gate_c_facet_by_scale_matrix.csv",
                      "fdr_exploratory_family_E3toE7.csv"]
print(f"{len(CHAIN)} expected outputs, across "
      f"{len(set(c[0] for c in CHAIN))} notebooks; {len(HUMAN_FILES)} of them "
      f"belong to the optional arm.")

In [ ]:
# === LOAD (3 of 3): the manifest, and what is actually in the folder ===
# WHAT THIS CELL DOES: reads _manifest.json, lists the working folder, and
# defines the two lookups the rest of the notebook uses.
#
# DISK IS THE FACT, THE MANIFEST IS THE STORY. A file recorded in the manifest
# but since deleted would be reported as present and then fail to copy; a file
# dropped into the folder by hand is real even though no notebook recorded it.
# So presence is decided by listing the folder, and the manifest is used only for
# who wrote a file, when, and in which run mode.
import pandas as pd
import shutil, zipfile

M       = _load_manifest()
RUNS    = {r["nb"]: r for r in M.get("runs", [])}   # notebook id -> its run record
RECORDS = M.get("files", {})                        # file name -> what wrote it
ON_DISK = sorted(f for f in os.listdir(WORKDIR)
                 if os.path.isfile(os.path.join(WORKDIR, f)))

def here(name):
    """True when the file is in the working folder."""
    return os.path.exists(os.path.join(WORKDIR, name))

def matching(prefix):
    """Every file whose name starts with prefix.

    The per-model scores and the per-encoder embedding caches carry the model or
    encoder key in the file name, so how many there are and what they are called
    is only knowable by looking. Matching on the prefix is what lets the report
    say "three models scored" without the model set being hard-coded here and
    drifting away from the notebook that actually chose it.
    """
    return [f for f in ON_DISK if f.startswith(prefix)]

print(f"manifest: {len(RECORDS)} files recorded, {len(RUNS)} notebook runs.")
print(f"folder:   {len(ON_DISK)} files in {WORKDIR}")

## Do

In [ ]:
# === DO (1 of 10): which notebooks ran, in what mode, and when ===
# THIS IS THE PRINTOUT TO READ FIRST. Every notebook stamps its number, its run
# mode and the time into the manifest as it finishes. Reading them back in order
# is the state of the whole project on one screen.
print("STATE OF THE CHAIN")
print(f"  {'nb':<4}{'job':<56}{'ran':<9}{'mode':<7}when")
print("  " + "-" * 96)
ran_ids, smoke_ids, gold_flags = [], [], set()
for nb in sorted(NB_JOB):
    job = NB_JOB[nb][:55]
    if nb == NB_ID:                       # this notebook has not finished yet, so
        print(f"  {nb:<4}{job:<56}{'now':<9}{RUN_MODE:<7}running")
        continue                          # it has no manifest record to read back
    r = RUNS.get(nb)
    if r is None:
        print(f"  {nb:<4}{job:<56}{'-- no':<9}")
        continue
    ran_ids.append(nb)
    if r.get("run_mode") == "smoke":
        smoke_ids.append(nb)
    gold_flags.add(bool(r.get("human_gold")))
    print(f"  {nb:<4}{job:<56}{'yes':<9}{str(r.get('run_mode', '?')):<7}"
          f"{r.get('at', 'unknown time')}")

# WHAT THIS RUN NEEDS: every notebook but this one, and not the optional arm
# unless somebody asked for it. With HUMAN_GOLD False notebook 99 was never meant
# to run, and listing it as still to do would send the reader off to run a
# notebook the registration does not ask for.
#
# COUNTED AGAINST THE SAME LIST IT IS COMPARED TO. Both numbers on the line below
# come from EXPECTED, so they cannot disagree: a notebook that ran here but this
# run does not need -- the arm, on a registered run -- gets its own line instead
# of inflating the count into "12 of the 11 notebooks this run needs".
EXPECTED = [nb for nb in sorted(NB_JOB)
            if nb != NB_ID and not (nb == OPTIONAL_NB and not HUMAN_GOLD)]
NOT_RUN    = [nb for nb in EXPECTED if nb not in RUNS]
ran_needed = [nb for nb in ran_ids if nb in EXPECTED]
ran_spare  = [nb for nb in ran_ids if nb not in EXPECTED]
print(f"\n  ran: {len(ran_needed)} of the {len(EXPECTED)} notebooks this run needs"
      + (f"   still to run: {', '.join(NOT_RUN)}" if NOT_RUN
         else "   still to run: none"))
if ran_spare:
    print(f"  notebook(s) {', '.join(ran_spare)} also ran in this folder at some "
          f"point. This run does not use them.")

In [ ]:
# === DO (2 of 10): the two ways a chain can be quietly inconsistent ===
# Both of these produce files that look perfectly normal from the outside, which
# is exactly why they are checked here rather than left to be noticed. Both look
# at every notebook that ran, INCLUDING the optional arm: a smoke-mode run of 99
# is the one that would poison clause A, and it is the easiest to overlook.
#
# (1) A SMOKE-MODE LINK. A notebook run in smoke mode saw the first 20
#     PARTICIPANTS -- 60 answers, three each -- and not the corpus. The trim is by
#     participant and never by row, so its output has the same name and the same
#     columns as a full run's, with about a fiftieth of the rows, and it will
#     merge, correlate and export without complaint. A chain with one smoke-mode
#     link in it is not a result.
if smoke_ids:
    print("WARNING: these notebooks last ran in SMOKE mode, on the first 20 "
          "participants (60 answers, not 2,844): " + ", ".join(smoke_ids))
    print("  Anything computed from them is a plumbing test, not a finding.")
    print("  Re-run them with RUN_MODE = \"full\" before depositing anything.")
else:
    print("Run mode: every notebook that ran, ran on the full corpus.")

# (2) A MIXED HUMAN_GOLD CHAIN. HUMAN_GOLD decides whether the human-coder arm
#     runs at all, and the gate notebook reports clause A on that basis. If some
#     notebooks ran with it True and others False, the gate table can carry a
#     clause A computed on a gold set that a later notebook then ignored, and the
#     printed gate mode would describe neither run. The fix is to settle the
#     switch and re-run, not to reason about which half won.
if len(gold_flags) > 1:
    print("\nWARNING: the notebooks did not agree on HUMAN_GOLD -- some ran with")
    print("  the human-coder arm on and some with it off. Clause A in the gate")
    print("  table may describe a run that no longer exists. Settle the switch")
    print(f"  and re-run the notebooks from {OPTIONAL_NB} onward.")
elif gold_flags:
    only = "on" if True in gold_flags else "off"
    print(f"Human-coder arm: every notebook ran with HUMAN_GOLD {only}.")
    if only == "off":
        print("  That is the registered pipeline (prereg 6.7). Clause A of the")
        print("  gate is our addition, and it is reported as not run, never as a")
        print("  failure -- a check nobody performed did not fail.")

In [ ]:
# === DO (3 of 10): which outputs are here and which are not ===
# WHAT THIS CELL DOES: walks the expected-output table, says whether each file is
# in the folder, and counts what is missing.
#
# A MISSING "human" ROW IS NOT A MISSING FILE when HUMAN_GOLD is False. Those
# three files come from the optional arm, which the registered pipeline does not
# run, so they are reported as "not run" rather than as absent. The difference
# matters: absent means something failed, not run means nobody asked for it.
#
# BUT THE FOLDER IS STILL SEARCHED. "not run" is the wrong word for a file that is
# sitting right there from an earlier pass with the arm switched on. That file is
# a deviation from the registration, it must not travel inside a run that reports
# itself as registered, and saying "not run" over the top of it is how it travels.
print("EXPECTED OUTPUTS")
print(f"  {'nb':<4}{'file':<40}{'status':<12}what its absence costs")
print("  " + "-" * 100)
missing_core, missing_extra, stale_human = [], [], []
for nb, name, kind, cost in CHAIN:
    shown = name + "*" if kind == "many" else name
    if kind == "many":
        hits = matching(name)
        status = f"{len(hits)} file(s)" if hits else "MISSING"
    elif kind == "human" and not HUMAN_GOLD:
        status = "LEFT OVER" if here(name) else "not run"
        if here(name):
            stale_human.append(name)
    else:
        status = "present" if here(name) else "MISSING"
    if status == "MISSING":
        # A "human" row counts as needed only when the arm was asked for. With
        # HUMAN_GOLD on and the file absent, the arm was switched on and produced
        # nothing, which is a real gap rather than an optional one.
        needed = kind in ("core", "many") or (kind == "human" and HUMAN_GOLD)
        (missing_core if needed else missing_extra).append(
            f"{shown} (notebook {nb})")
    print(f"  {nb:<4}{shown:<40}{status:<12}{cost[:52]}")

print(f"\n  missing and needed: "
      + (", ".join(missing_core) if missing_core else "nothing"))
print(f"  missing but optional: "
      + (", ".join(missing_extra) if missing_extra else "nothing"))
if stale_human:
    print(f"  LEFT OVER from a run with the arm on: {', '.join(stale_human)}.")
    print(f"  This run has HUMAN_GOLD False, so notebook {OPTIONAL_NB} is not part")
    print("  of it. These files are kept out of for_rmd/ below and named again in")
    print("  the OSF list. Move them aside, or post the amendment and re-run with")
    print("  HUMAN_GOLD True, before this bundle goes anywhere.")

In [ ]:
# === DO (4 of 10): anything in the folder the table does not know about ===
# WHY THIS EXISTS. The expected-output table above is written down by hand, and a
# notebook upstream can rename a file without this one being told. If that
# happens the file shows up here as an extra, instead of disappearing from the
# report while its old name is announced as missing. Two lines of a listing are a
# cheap price for never silently losing an output.
known = set()
for nb, name, kind, cost in CHAIN:
    if kind == "many":
        known.update(matching(name))
    else:
        known.add(name)
# Files this notebook itself makes are not surprises, so they are not listed.
mine = {"_manifest.json", "osf_bundle_manifest.json", "retribution_nlp_bundle.zip"}
extras = [f for f in ON_DISK if f not in known and f not in mine]
if extras:
    print("ALSO IN THIS FOLDER (not in the expected-output table above):")
    for f in extras:
        rec = RECORDS.get(f, {})
        by = rec.get("written_by", "NOT RECORDED IN THE MANIFEST")
        print(f"  {f:<44}{by}")
    print("\n  If one of these is an expected output under a new name, the row")
    print("  above that reads MISSING is stale rather than true.")
    print("  A file with a notebook beside it travels in the bundle. A file that")
    print("  reads NOT RECORDED does NOT: the bundle is built from the manifest,")
    print("  so nothing vouches for a file no notebook registered. If it belongs")
    print("  in the deposit, find the notebook that should have written it.")
else:
    print("Nothing in the folder that the expected-output table does not cover.")

In [ ]:
# === DO (5 of 10): the six names the R Markdown reads ===
# THE NAMES ON THE LEFT ARE NOT NEGOTIABLE. 02_nlp_integration.Rmd looks for
# exactly these six file names on its search path (Analysis/NLP/outputs, then
# Analysis/NLP/colab, then R/nlp, then R/data/nlp) and takes the first hit. Any
# it cannot find, it replaces with a boxed "AWAITING COLAB OUTPUTS" notice and
# carries on. So a file under a slightly different name does not produce a
# smaller version of that section -- it produces an empty one, and the reader of
# the knitted document sees a placeholder where a table should be and has no way
# to tell a naming slip from an analysis that was never run.
#
# Three of the six already carry the right name and are copied unchanged. Three
# are this chain's files under the R Markdown's name: the mapping is the whole
# content of this cell, and it is the one place the two halves of the project
# agree on vocabulary.
FOR_RMD = "for_rmd"        # the folder to drop into Analysis/NLP/outputs/
os.makedirs(FOR_RMD, exist_ok=True)

RMD_INPUTS = {
    # name the R Markdown reads            # this chain's file       # notebook
    "ensemble_facet_scores.csv":      ("ensemble_scores.csv",           "07"),
    "ensemble_agreement.csv":         ("inter_model_icc.csv",           "07"),
    "embedding_facet_similarity.csv": ("facet_similarity.csv",          "09"),
    "facet_dictionary_scores.csv":    ("facet_dictionary_scores.csv",   "01"),
    "gold_codes.csv":                 ("gold_codes.csv",                "99"),
    "gold_split_assignment.csv":      ("gold_split_assignment.csv",     "03"),
}

# THE FOLDER IS EMPTIED FIRST. for_rmd/ is not a record of anything. It is a
# staging area that gets dropped whole into Analysis/NLP/outputs/, and the R
# Markdown reads whatever it finds there as this run's output. A file left behind
# by an earlier run is therefore indistinguishable from one this run wrote, and
# the run that suffers is the registered one: a gold_codes.csv from a pass with
# HUMAN_GOLD True is human coding arriving inside a run that reports itself as
# models-only. Clearing the folder is what makes "5 of 6 written" and the contents
# of the folder the same statement.
_cleared = [f for f in sorted(os.listdir(FOR_RMD))
            if os.path.isfile(os.path.join(FOR_RMD, f))]
for _f in _cleared:
    os.remove(os.path.join(FOR_RMD, _f))
if _cleared:
    print(f"Cleared {len(_cleared)} file(s) an earlier run left in {FOR_RMD}/: "
          + ", ".join(_cleared))

In [ ]:
# === DO (5 of 10, cont.): copy them, under R's names ===
# TWO LINES PER FILE. The [ok] line is need(), reporting the chain's own file and
# when the chain wrote it -- called with optional=True, so a missing input costs
# one printed line and stops nothing. The [copy] line is what went into for_rmd/
# under the name R reads it by.
print("THE SIX FILES 02_nlp_integration.Rmd READS")
written_for_rmd = []
for rmd_name, (src_name, nb) in RMD_INPUTS.items():
    if src_name in HUMAN_FILES and not HUMAN_GOLD:
        # Not "is it here" but "was it asked for". A leftover gold_codes.csv is on
        # disk and would copy perfectly; it is refused because this run is the
        # registered pipeline and the arm is not part of it.
        print(f"  [skip] {rmd_name:<32} the human arm did not run "
              f"(HUMAN_GOLD is False)")
        continue
    src = need(src_name, produced_by=nb, optional=True)
    if src is None:
        print(f"  [skip] {rmd_name:<32} no {src_name} -- run notebook {nb}")
        continue
    dest = os.path.join(FOR_RMD, rmd_name)
    # copyfile rather than rename: the chain's own name has to survive, because
    # every other notebook reads the file under it.
    shutil.copyfile(src, dest)
    written_for_rmd.append(rmd_name)
    same = "" if src_name == rmd_name else f"  (from {src_name})"
    print(f"  [copy] {rmd_name:<32} {os.path.getsize(dest):>10,} bytes{same}")
print(f"\n  {len(written_for_rmd)} of {len(RMD_INPUTS)} written into {FOR_RMD}/")

In [ ]:
# === DO (6 of 10): what R will look for in each of those files ===
# WHY THIS CHECK IS HERE AND NOT IN R. The R Markdown does not assume column
# names, it discovers them: it accepts any of four names for the ensemble score
# column and any of eight for the intraclass correlation. That tolerance is
# useful, but it has one edge -- when it finds NONE of the candidates it clears
# its readiness flags and prints the same awaiting-Colab box as a missing file.
# A column-name slip and an un-run notebook then look identical from the R side.
# Checking here names the problem on the side that made the file.
WANTED = {
    "ensemble_facet_scores.csv":
        (["response_id", "item_type", "facet"],
         ["ensemble_mean", "facet_score", "score", "mean"]),
    "ensemble_agreement.csv":
        (["facet"],
         ["icc2k", "icc_2k", "ICC2k", "ICC_2k", "icc", "ICC", "icc2_1", "icc21"]),
    "embedding_facet_similarity.csv":
        (["response_id", "item_type", "facet"], None),   # one *_sim per encoder
    "facet_dictionary_scores.csv":
        (["response_id", "item_type", "n_words"], None),
    "gold_codes.csv":
        (["response_id", "item_type"], None),
    "gold_split_assignment.csv":
        (["response_id", "item_type"], ["gold_split", "split", "assignment"]),
}

# AND THE KEY CONTRACT, WHICH R DOES NOT CHECK AT ALL. The corpus is 948
# participants x 3 open-ended answers = 2,844 rows, keyed on response_id x
# item_type, and on the facet-long files x facet as well. A repeated key is the
# one defect that survives every check downstream: 02_nlp_integration.Rmd
# reshapes the ensemble file with values_fn = mean, which averages a repeat
# instead of objecting, and a repeat in the dictionary or split file fans out
# through the join and gives that participant double weight in every correlation
# while the printed sample size does not move. This is the last place it can be
# caught before the files leave the chain.
KEYED = {
    "ensemble_facet_scores.csv":      ["response_id", "item_type", "facet"],
    "ensemble_agreement.csv":         ["facet"],
    "embedding_facet_similarity.csv": ["response_id", "item_type", "facet"],
    "facet_dictionary_scores.csv":    ["response_id", "item_type"],
    "gold_codes.csv":                 ["response_id", "item_type"],
    "gold_split_assignment.csv":      ["response_id", "item_type"],
}

def read_staged(name):
    """Read one staged file, or return None and say why.

    A zero-byte file -- a notebook killed part-way through its write -- comes out
    of read_csv as EmptyDataError, and a half-written one as a parser error.
    Neither may stop this notebook: it is the one that has to survive a broken
    chain and report it, and a defect in a file it merely copies must not take
    the state report and the bundle down with it.
    """
    try:
        return pd.read_csv(os.path.join(FOR_RMD, name))
    except Exception as e:
        print(f"  [!!]   {name:<32} cannot be read: {type(e).__name__}")
        return None

In [ ]:
# === DO (6 of 10, cont.): run the two contracts over the staged files ===
print("COLUMN AND KEY CONTRACT")
contract_problems, key_problems, unreadable = [], [], []
for name in written_for_rmd:
    must, one_of = WANTED[name]
    frame = read_staged(name)
    if frame is None:
        contract_problems.append(f"{name}: could not be read")
        unreadable.append(name)
        continue
    cols = list(frame.columns)
    gaps = [c for c in must if c not in cols]
    if one_of is not None and not any(c in cols for c in one_of):
        gaps.append("one of " + "/".join(one_of[:4]))
    if name == "embedding_facet_similarity.csv" and not [
            c for c in cols if c.endswith("_sim")]:
        gaps.append("at least one *_sim column")
    keys = [k for k in KEYED[name] if k in cols]
    # No key columns, nothing to be unique on -- and R discards a file whose key
    # columns are missing, so that case is the column contract's above and does
    # not stop anything here.
    dup = int(frame.duplicated(keys).sum()) if len(keys) == len(KEYED[name]) else 0
    if dup:
        key_problems.append(f"{name}: {dup} repeated {' x '.join(keys)} key(s)")
    if gaps:
        contract_problems.append(f"{name}: no {', '.join(gaps)}")
        print(f"  [!!]   {name:<32} missing {', '.join(gaps)}")
    elif dup:
        print(f"  [!!]   {name:<32} {len(frame):,} rows, {dup} REPEATED KEYS")
    else:
        shape = f"{len(frame):,} rows"
        if "response_id" in cols:
            shape += f", {frame['response_id'].nunique():,} participants"
        print(f"  [ok]   {name:<32} {len(cols)} columns, {shape}, "
              f"unique on {' x '.join(keys)}")
# A STAGED COPY THAT CANNOT BE READ IS TAKEN BACK OUT of for_rmd/. Its source is
# broken -- a notebook killed part-way through its write leaves a zero-byte file --
# and the honest thing to hand R is nothing at all: a file R cannot find prints
# the awaiting-Colab box, which says the analysis has not arrived, while an
# unreadable file sitting there is read as an analysis that arrived empty.
for name in unreadable:
    os.remove(os.path.join(FOR_RMD, name))
    written_for_rmd.remove(name)
    print(f"  [--]   {name:<32} taken back out of {FOR_RMD}/. Re-run the "
          f"notebook that writes {RMD_INPUTS[name][0]} "
          f"(notebook {RMD_INPUTS[name][1]}).")
if contract_problems:
    print("\n  A file above will be READ BY R AND THEN DISCARDED. Fix the")
    print("  notebook that writes it; renaming the column by hand puts the")
    print("  chain and the document out of step at the next re-run.")
if key_problems:
    print("\n  A REPEATED KEY IS THE SERIOUS ONE: R will not discard that file, it")
    print("  will average the repeat and report the sample size it always did.")
    print("  The export below refuses to deposit it, and says which notebook to")
    print("  fix. Nothing about it can be corrected in this notebook.")

In [ ]:
# === DO (7 of 10): what the gate said ===
# WHAT THIS CELL DOES: reads gate_results.csv back and prints it. It decides
# nothing -- the gate is settled once, in the gate notebook, and re-deciding it
# here is how two copies of a rule drift apart.
#
# THE REGISTERED GATE HAS TWO CLAUSES. Prereg 6.7: a facet supports a claim only
# if the models agree at an intraclass correlation of at least .60 (clause B) and
# the facet correlates at least .30 with its matching survey scale (clause C).
# Clause A, agreement with human coders, is OUR ADDITION and a registered
# deviation; it runs only when HUMAN_GOLD is True.
#
# GATE-BACKED IS NOT CONFIRMATORY. Every E3 to E7 test is exploratory (prereg
# 2.4). What the gate decides is whether a facet is a primary measure or a
# descriptive one INSIDE that exploratory family, never whether a test is
# confirmatory.
def clause(v):
    """Read one gate cell as a word, so a blank can never be read as a failure.

    A gate table can carry a word ("pass" / "fail" / "not run") or a True/False,
    depending on which notebook wrote the column, and a clause that was never
    evaluated arrives as an empty cell. All three have to come out as words,
    because bool(None) is False: left alone, a clause nobody ran would print as
    a clause that failed, and a reader skimming the table would strike a facet
    that was never tested.
    """
    if v is None or (not isinstance(v, str) and pd.isna(v)) or str(v).strip() == "":
        return "not run"
    s = str(v).strip()
    if s.lower() in ("true", "1", "1.0"):
        return "pass"
    if s.lower() in ("false", "0", "0.0"):
        return "fail"
    return s

def text(v):
    """Print an empty cell as empty. Left alone, str() of a missing value in a
    spreadsheet column is the word 'nan', which is not a reason and not a
    standing, and looks like something rather than like nothing."""
    return "" if v is None or (not isinstance(v, str) and pd.isna(v)) else str(v)

In [ ]:
# === DO (7 of 10, cont.): read the table, and the three words it may carry ===
# THE THREE STANDINGS notebook 10 writes into the status column. Compared after
# the separator is normalised, because an exact string match on "GATE_BACKED" is
# a hyphen away from reading three gate-backed facets as none and printing "no
# facet cleared the gate" underneath a table that says otherwise. Anything not on
# this list is reported as a word this notebook does not recognise -- which is
# not the same statement as "the gate failed", and must never be printed as one.
GATE_STATUSES = ("GATE_BACKED", "DESCRIPTIVE_ONLY", "CONVERGENT_SANITY")

def standing_word(v):
    """The status cell with hyphens and spaces read as underscores, upper case."""
    return text(v).strip().upper().replace("-", "_").replace(" ", "_")

GATE_BACKED, GATE_STATE = [], "not run"
gate_table, gate_absent = None, "gate_results.csv is not in the folder"
_gate_path = need("gate_results.csv", produced_by="10", optional=True)
if _gate_path:
    try:
        gate_table = pd.read_csv(_gate_path)
    except Exception as _e:
        # A gate table that cannot be parsed says exactly as much about a facet as
        # a gate table that is not there: nothing. It is reported the same way and
        # it does not stop the export, which has a bundle to write either way.
        gate_absent = f"gate_results.csv cannot be read ({type(_e).__name__})"

In [ ]:
# === DO (7 of 10, cont.): print the gate, facet by facet ===
unknown_standing = []
if gate_table is None or len(gate_table) == 0:
    # An empty table and an absent one are reported the same way and for the same
    # reason: neither one says a facet failed, both say nobody has decided yet.
    GATE_STATE = ("not run -- " + gate_absent if gate_table is None
                  else "not run -- gate_results.csv is empty")
    print(f"THE GATE HAS NOT RUN ({GATE_STATE}), so no facet has")
    print("standing yet and every facet score in the bundle is provisional.")
    print("Run notebook 10. Until then the R Markdown will print its gate section")
    print("as an awaiting-Colab notice, which is the correct reading, not a bug.")
else:
    cols = list(gate_table.columns)
    mode = (text(gate_table["gate_mode"].iloc[0]) if "gate_mode" in cols else "")
    if not mode:   # the table did not record it, so say which gate this run could run
        mode = "registered + human (A+B+C)" if HUMAN_GOLD else "registered (B+C)"
    status_col = "status" if "status" in cols else (
        "standing" if "standing" in cols else None)
    print(f"THE GATE, as notebook 10 left it.  Mode: {mode}")
    print(f"  {'facet':<18}{'A':<10}{'B':<10}{'C':<10}{'standing':<20}why")
    print("  " + "-" * 96)
    for _, r in gate_table.iterrows():
        a, b, c = (clause(r.get("clauseA")), clause(r.get("clauseB")),
                   clause(r.get("clauseC")))
        st = text(r.get(status_col)) if status_col else ""
        why = text(r.get("reason"))[:34]
        print(f"  {text(r.get('facet')) or '?':<18}{a:<10}{b:<10}{c:<10}{st:<20}{why}")
        w = standing_word(st)
        if w == "GATE_BACKED":
            GATE_BACKED.append(text(r.get("facet")))
        elif w and w not in GATE_STATUSES:
            unknown_standing.append(st)

In [ ]:
# === DO (7 of 10, cont.): and what that adds up to ===
if gate_table is not None and len(gate_table):
    if status_col is None:
        GATE_STATE = ("cannot be read: gate_results.csv has no status column, so "
                      "no facet's standing is stated in it")
    elif unknown_standing:
        GATE_STATE = ("cannot be read: the status column holds "
                      + ", ".join(sorted(set(unknown_standing))[:3])
                      + ", which this notebook does not recognise")
    elif GATE_BACKED:
        GATE_STATE = f"{len(GATE_BACKED)} facet(s) gate-backed: " + ", ".join(GATE_BACKED)
    else:
        GATE_STATE = "no facet cleared the gate; every facet is descriptive-only"
    print(f"\n  {GATE_STATE}")
    if status_col is None or unknown_standing:
        # Not a verdict. A standing this notebook cannot parse is a reading
        # problem, and printing it as "no facet cleared the gate" would turn a
        # vocabulary mismatch into a finding about the facets.
        print("  That is a reading problem, not a gate verdict. Notebook 10 writes")
        print(f"  one of {', '.join(GATE_STATUSES)} into the status column; check")
        print("  what wrote this table before quoting anything from it.")
    print("  Gate-backed is not confirmatory: E3 to E7 are exploratory (prereg 2.4),")
    print("  and the gate only says which facets are primary inside that family.")
    if not HUMAN_GOLD:
        print("  Clause A reads 'not run' on every row because the human arm was")
        print("  switched off. That is the registered design, not a failure.")

In [ ]:
# === DO (8 of 10): the prosocial-minus-dark footnote ===
# The last surviving trace of the Dark Side pipeline's prosocial-versus-dark
# contrast, which the nine-facet scheme replaced (prereg 5.15: the categories
# change "from a prosocial-versus-dark split to the retributive facets"). The
# companion "% closer to dark" metric was dropped outright -- it swung from 67.5%
# to 21-28% across encoders, so only the rank ordering ever survived a change of
# embedding model. This roll-up is kept as ONE descriptive line and nothing
# routes through it.
#
# The try/except is deliberate: a footnote must never be able to abort the export
# around it.
try:
    pro  = ["deterrence", "incapacitation", "rehabilitation"]  # consequentialist
    dark = ["revenge", "suffering"]                            # vengeful
    _ens_path = need("ensemble_scores.csv", produced_by="07", optional=True)
    if _ens_path:
        ens = pd.read_csv(_ens_path)
        # pivot_table AVERAGES a repeated key rather than raising, so a response
        # that appears twice would slide into the mean unannounced. One line to
        # say so, and the footnote stands aside rather than quote a number built
        # on a corpus that is not the corpus.
        _dup = int(ens.duplicated(["response_id", "item_type", "facet"]).sum())
        wide = ens.pivot_table(index=["response_id", "item_type"],
                               columns="facet", values="ensemble_mean")
        missing = [c for c in pro + dark if c not in wide.columns]
        if _dup:
            print(f"[footnote] prosocial-minus-dark not computed: "
                  f"ensemble_scores.csv repeats {_dup} response x item x facet "
                  f"key(s), so the mean would be taken over a corpus that is not "
                  f"the corpus.")
        elif not missing:
            # All five columns or nothing: a mean over a partial set would not be
            # the same construct, and a footnote computed on three of five facets
            # under the same name is worse than no footnote.
            pmd = wide[pro].mean(axis=1) - wide[dark].mean(axis=1)
            print(f"[footnote] prosocial-minus-dark roll-up (DESCRIPTIVE ONLY): "
                  f"mean={pmd.mean():.3f}, sd={pmd.std():.3f} -- retired "
                  f"construct, not an analysis.")
        else:
            print(f"[footnote] prosocial-minus-dark not computed: "
                  f"{', '.join(missing)} not scored.")
    else:
        print("[footnote] prosocial-minus-dark not computed: no ensemble scores.")
except Exception as e:
    print(f"[footnote] prosocial-minus-dark not computed: "
          f"{type(e).__name__}: {e}")

In [ ]:
# === DO (9 of 10): the OSF bundle manifest ===
# WHY THE MANIFEST EXISTS AS A FILE. Prereg 6.7 commits the project to a specific
# deposit: "The model set and versions, the scoring prompts, the word lists, the
# raw model output, and the analysis code go on the project's OSF page." Those
# artifacts are produced across a dozen notebooks, and a written list that
# travels with the results is what makes an incomplete deposit visible instead of
# silent. Each key answers one reproducibility question: what were the models
# told, which models exactly, what did the non-model check consist of, what did
# the models literally return, which half of the corpus was allowed to seed
# anything, and which facets earned primary standing.
#
# Every technical term is glossed where it appears, because this list is printed
# to the screen below and it is what a reader of the deposit reads first.
OSF_BUNDLE = {
    "prompts_and_rubric":
        "rubric.json -- the annotator instruction, the nine facet definitions and "
        "the strict JSON reply schema, with the fingerprint every rater was "
        "checked against (nb 05)",
    "pinned_model_versions":
        "the model registry: the dated model string for every rater, recorded on "
        "each line of the raw output and in the manifest note (nb 06)",
    "word_lists":
        "facet_dictionary_scores.csv plus the nine regular-expression buckets, "
        "one per facet -- the patterns a word is matched against (nb 01) -- and "
        "word_list_coverage.csv, the 5% coverage rule applied (nb 02)",
    "embedding_reference_sets":
        "embedding_seed_pools.json -- the codebook's written anchor passages "
        "plus dictionary-positive SEED-half responses (nb 04)",
    "raw_per_model_output":
        "llm_scores_<model>.jsonl -- every call exactly as the model returned it, "
        "which is the verbatim record prereg 6.7 commits to. "
        "llm_scores_<model>.csv is the same content parsed into a table (nb 06)",
    "frozen_split":
        "gold_split_assignment.csv -- SEED/TEST, frozen from RANDOM_SEED "
        "before any scoring (nb 03)",
}

In [ ]:
# === DO (9 of 10, cont.): the results half of the list ===
OSF_BUNDLE.update({
    "inter_model_agreement":
        "inter_model_icc.csv -- gate clause B. The intraclass correlation, a "
        "measure of how much independent raters agree, for the ensemble as a "
        "whole (ICC(2,k)) and for one average rater (ICC(2,1)); plus "
        "Krippendorff's alpha, an agreement measure that discounts the agreement "
        "raters would reach by chance (nb 07)",
    "stream_agreement":
        "stream_agreement_matrix.csv -- the model, word-list and embedding "
        "streams set against each other. This is the table that answers whether "
        "a facet is measured or only asserted (nb 10)",
    "gate_table":
        "gate_results.csv -- the clauses per facet and each facet's standing; "
        "and gate_c_facet_by_scale_matrix.csv, the whole clause-C matrix of each "
        "facet against its own survey scale and against its rivals (nb 10)",
    "results":
        "E3_to_E7_results.csv and fdr_exploratory_family_E3toE7.csv (nb 11). E3 "
        "to E7 are ONE exploratory family, corrected together by the "
        "Benjamini-Hochberg procedure, which holds down the share of false "
        "positives among the results called significant, per prereg 2.4 and 6.7",
    "human_gold_and_irr":
        ("gold_codes.csv, the adjudicated human codes, and gold_irr_by_facet.csv: "
         "per-facet AC1 and Cohen's kappa, two measures of how often two coders "
         "gave the same answer once chance agreement is taken out, plus "
         "Krippendorff's alpha, the same idea across every coder at once (nb 99)"
         if HUMAN_GOLD else
         "NOT RUN. Prereg 6.7 has the ensemble code the corpus in place of "
         "human raters and prescribes no human-coding step; the human arm and "
         "gate clause A are our addition, and this run did not use them."),
    # The deviations belong in the bundle as much as the results do: the
    # registration is approved and out of embargo, so each departure goes up as a
    # DATED amendment posted before the ensemble scores the corpus, which is what
    # keeps any of them from being read as post-hoc.
    "prereg_deviations":
        "MEASUREMENT_DESIGN.md section 0 -- the deviations table (human gold "
        "standard and gate clause A, proportionality demotion, word-list "
        "coverage rule, tightened clause C, CI-bound gating, E4 split-call) and "
        "the dated OSF amendments they require",
})
# The list must not say NOT RUN while the arm's output sits in the folder. It was
# kept out of for_rmd/, but it is still on disk and the bundle is built from the
# manifest, so it would travel inside the archive under a registered label.
if not HUMAN_GOLD and stale_human:
    OSF_BUNDLE["human_gold_and_irr"] += (
        "  IN THE FOLDER ANYWAY: " + ", ".join(sorted(stale_human)) + ", from an "
        "earlier pass with the arm on. Take these out of the deposit, or post the "
        "amendment and re-run with HUMAN_GOLD True, before this bundle goes up.")

In [ ]:
# === DO (9 of 10, cont.): write the list and say what carries text ===
with open("osf_bundle_manifest.json", "w") as fh:
    json.dump(OSF_BUNDLE, fh, indent=2)
print("OSF BUNDLE MANIFEST")
for k, v in OSF_BUNDLE.items():
    print(f"  {k}\n      {v}")

# THE ONE THING TO DO BEFORE ANY OF THIS IS POSTED. Several files in the bundle
# carry participant writing word for word, not just numbers derived from it: the
# corpus itself, the raw model output and the smoke test (both of which can echo
# quoted spans), the reference passages (which ARE participant sentences), the
# sampling frame and, where the human arm ran, the gold codes. Posting a bundle is
# publishing that text under an open licence, and it cannot be recalled.
print("\nDE-IDENTIFICATION: these files carry verbatim participant text and must")
print("be cleared before anything is posted --")
for f in ["openends_long.csv", "strata_sample.csv", "embedding_seed_pools.json",
          "smoke_test_results.csv", "gold_codes.csv"] + matching("llm_scores_"):
    if here(f):
        print(f"    {f}")

In [ ]:
# === DO (10 of 10): the five tables the deposit cannot go up without ===
# WHY THESE FIVE. Each one is written by a single cell at the end of a notebook,
# so a gap here means that cell did not complete -- most often because the run was
# partial or was stopped. Between them they hold every number a reader would
# quote: the gate, the results, the corrected family, the agreement between the
# streams and the discriminant matrix behind clause C. The bundle is written
# either way, because a partial bundle is useful for carrying on with. It is the
# DEPOSIT that must not go up with a NO in this list.
print("THE FIVE THE DEPOSIT MUST CARRY")
deposit_gaps = [f for f in DEPOSIT_MUST_CARRY if not here(f)]
for f in DEPOSIT_MUST_CARRY:
    print(f"  {'yes' if here(f) else 'NO':<5}{f}")
if deposit_gaps:
    print(f"\n  {len(deposit_gaps)} of the five are not here. What follows is an")
    print("  archive, not a deposit: post it and the gap goes up with it. Run the")
    print("  notebook that owes each one -- the table further up says which.")
else:
    print("\n  All five are here. On this count the deposit is complete.")

## Save and check

In [ ]:
# === SAVE (1 of 3): register the six copies and the OSF list ===
# THE ORDER MATTERS. Everything is registered with done() BEFORE the archive is
# built, so the _manifest.json that travels inside the archive is the one that
# describes it. Registered afterwards, the deposited manifest lists none of the
# six copies, and a reader who opens the archive finds files that on the
# manifest's own evidence never happened. The copies are registered under their
# for_rmd/ path, so the manifest records both names -- the chain's and R's -- for
# the same data.
#
# THE ASSERT IS THE ONE HARD STOP IN THIS NOTEBOOK, and it is deliberate. Every
# INPUT here is looked up rather than demanded, because this notebook has to be
# able to report on a half-finished chain. Its own OUTPUT is a different matter.
# A file leaving with a repeated key gives one participant double weight in every
# correlation R computes from it, with the sample size printed unchanged, and R
# will not catch it: the reshape averages repeats rather than objecting. Refusing
# to register the copy is the last chance to stop that, and the run stops with the
# name of the notebook that has to be fixed. Everything above this line has
# already printed, so the state report survives the stop.
for name in list(written_for_rmd):
    p = os.path.join(FOR_RMD, name)
    frame = read_staged(name)
    if frame is None:
        # Readable one cell ago and not now. Nothing is registered and the copy
        # goes, so the deposit never carries a file this notebook cannot vouch for.
        os.remove(p)
        written_for_rmd.remove(name)
        continue
    keys = [k for k in KEYED[name] if k in frame.columns]
    dup = int(frame.duplicated(keys).sum()) if len(keys) == len(KEYED[name]) else 0
    assert dup == 0, (
        f"\n\n  NOT DEPOSITED: {name} carries {dup} repeated "
        f"{' x '.join(keys)} key(s).\n"
        f"  One row per key is what every sample size in the R Markdown assumes, "
        f"and it averages\n  a repeat instead of raising, so this would arrive as "
        f"a wrong number that looks right.\n"
        f"  Fix the notebook that writes {RMD_INPUTS[name][0]} "
        f"(notebook {RMD_INPUTS[name][1]}), then run this notebook again.\n")
    done(p, rows=len(frame),
         note=f"copy of {RMD_INPUTS[name][0]} under the name "
              f"02_nlp_integration.Rmd reads")
done("osf_bundle_manifest.json", rows=len(OSF_BUNDLE),
     note="what the OSF deposit must contain, per prereg 6.7")

In [ ]:
# === SAVE (2 of 3): the archive bundle ===
# WHAT GOES IN: everything the chain recorded in the manifest and that is still on
# disk, the six R Markdown inputs, the manifest itself and the OSF list. Built
# from the manifest rather than from a written list of names, so a file this
# notebook has never heard of still travels, as long as the notebook that made it
# registered it with done(). The other side of that rule is in the extras block
# above: a file nobody registered does not travel.
#
# THE MANIFEST IS RE-READ HERE, not taken from the copy loaded at the top, because
# the six copies were registered one cell ago and belong in the archive's own
# record of itself.
#
# THE EMBEDDING CACHES GO IN TOO. They are the largest files in the folder by far
# and they are also the most expensive to reproduce -- rebuilding them means
# paying for the encoder runs again -- so the bundle keeps them.
BUNDLE = "retribution_nlp_bundle.zip"
bundle_files = []
for name in sorted(_load_manifest()["files"]):     # everything the chain wrote
    if here(name) and name != BUNDLE and name not in bundle_files:
        bundle_files.append(name)
for name in sorted(os.listdir(FOR_RMD)):           # the six, under R's names
    p = os.path.join(FOR_RMD, name)
    if p not in bundle_files:
        bundle_files.append(p)
for name in ["_manifest.json", "osf_bundle_manifest.json"]:
    if here(name) and name not in bundle_files:
        bundle_files.append(name)

# The run mode is in the folder name inside the zip so that a smoke-mode bundle
# announces itself on the way out of the archive, not after someone has analysed
# 20 participants' worth of answers.
ROOT = f"retribution_nlp_{RUN_MODE}"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    for f in bundle_files:
        z.write(f, arcname=os.path.join(ROOT, f))
print(f"{BUNDLE}: {len(bundle_files)} files, "
      f"{os.path.getsize(BUNDLE):,} bytes compressed.")
for f in bundle_files:
    print(f"    {f}")

In [ ]:
# === SAVE (2 of 3, cont.): read the archive back and check it ===
# TWO THINGS THIS NOTEBOOK COULD GET WRONG, both checked against the archive as
# written rather than against the list it was built from.
with zipfile.ZipFile(BUNDLE) as z:
    inside = set(z.namelist())

# (1) EVERYTHING PROMISED IS IN THERE. The three groups are gathered in different
# ways -- the copies from this run, the two manifests by name, the five result
# tables from the expected-output table -- so a bug in any one of those routes
# shows up here instead of at the moment somebody unzips the deposit.
promised = ([os.path.join(FOR_RMD, n) for n in written_for_rmd]
            + ["_manifest.json", "osf_bundle_manifest.json"]
            + [f for f in DEPOSIT_MUST_CARRY if here(f)])
missed = [f for f in promised if os.path.join(ROOT, f) not in inside]
assert not missed, (
    f"\n\n  The archive is missing {len(missed)} file(s) that this run wrote or "
    f"promised:\n    " + "\n    ".join(missed) + "\n"
    "  The bundle on disk is incomplete. Do not deposit it.\n")

# (2) THE DEVIATION DOES NOT TRAVEL IN A REGISTERED RUN. for_rmd/ is dropped whole
# into Analysis/NLP/outputs/ and read as this run's output, so a human-arm file in
# there is human coding inside a run that reports itself as models-only. The
# staging folder is cleared before the copies are made; this is the check that the
# clearing worked. Files at the top level of the folder are a different matter --
# they are the analyst's own data, they are named in the OSF list above, and this
# notebook reports them rather than deleting them.
leaked = sorted(f for f in inside
                if os.path.dirname(f) == os.path.join(ROOT, FOR_RMD)
                and os.path.basename(f) in HUMAN_FILES)
assert HUMAN_GOLD or not leaked, (
    f"\n\n  HUMAN_GOLD is False, and the archive still carries "
    f"{', '.join(os.path.basename(f) for f in leaked)} inside {FOR_RMD}/.\n"
    "  That folder is read by the R Markdown as this run's output, so the human "
    "arm would\n  enter a run that reports itself as the registered "
    "models-only pipeline. Delete the\n  file and run this notebook again.\n")
print(f"Archive checked: {len(promised)} promised file(s) all present"
      + ("; no human-arm file inside " + FOR_RMD + "/." if not HUMAN_GOLD
         else "; the human arm ran, so its files belong in there."))

In [ ]:
# === SAVE (3 of 3): register the archive, then the download ===
# The archive is registered last, and no archive can hold its own record: a
# file's fingerprint cannot be written into that same file once it is sealed. So
# on a first run the manifest inside the zip has no line for the bundle, and on a
# re-run it carries the previous bundle's line and the previous bundle's
# fingerprint. That single entry is the only way the manifest inside the archive
# differs from the one in the folder.
done(BUNDLE, rows=len(bundle_files), note=f"archive of the whole {RUN_MODE} chain")

# WHY A DOWNLOAD RATHER THAN A DRIVE MOUNT. The chain never calls drive.mount, so
# it does not depend on any particular Drive layout or on a Drive session staying
# authorized mid-run, and every artifact passes through the analyst's own hands on
# its way to the next stage -- which is also how the OSF deposit gets assembled.
# One zip rather than a burst of files: the browser blocks a burst until the tab
# is granted permission for multiple downloads, and one archive never trips it.
try:
    from google.colab import files as _colab_files
    _colab_files.download(BUNDLE)
    # "Download triggered" is not "download finished" -- check the browser's
    # download list before treating the run as complete.
    print(f"\nDownload of {BUNDLE} triggered. Check the browser's download list.")
except ImportError:
    print("\nNot running in Colab, so nothing was pushed to a browser.")
    print(f"  The bundle is at {os.path.join(WORKDIR, BUNDLE)}")
except Exception as e:
    print(f"\nCould not download {BUNDLE}: {e}")
    print(f"  It is on disk at {os.path.join(WORKDIR, BUNDLE)}")

In [ ]:
# === CHECK: what ran, what did not, what the gate said, what to do next ===
print("=" * 78)
print("WHERE THIS PROJECT IS")
print("=" * 78)
print(f"  run mode        {RUN_MODE}"
      + ("   -- SMOKE: 20 participants, 60 answers, a plumbing test"
         if RUN_MODE == "smoke" else ""))
arm = ("on -- clause A ran, and it is a registered deviation" if HUMAN_GOLD
       else "off -- the registered pipeline, clause A not run")
print(f"  human arm       {arm}")
print(f"  notebooks ran   {len(ran_needed)} of {len(EXPECTED)}"
      + (f"; still to run: {', '.join(NOT_RUN)}" if NOT_RUN else "; all of them"))
if smoke_ids:
    print(f"  SMOKE-MODE      {', '.join(smoke_ids)} -- re-run on the full corpus")
if missing_core:
    print(f"  missing files   {len(missing_core)}, and something needs each one:")
    for m in missing_core:
        print(f"                    {m}")
else:
    print("  missing files   none that anything needs")
if stale_human:
    print(f"  left over       {', '.join(stale_human)} -- the arm is off this run")
print(f"  the gate        {GATE_STATE}")
print(f"  for R           {len(written_for_rmd)} of {len(RMD_INPUTS)} files "
      f"written into {FOR_RMD}/")
if contract_problems:
    print(f"  column problem  {'; '.join(contract_problems)}")
if deposit_gaps:
    print(f"  deposit         NOT complete: {', '.join(deposit_gaps)} not written")

In [ ]:
# === CHECK, cont.: what to do next ===
print("\nWHAT TO DO NEXT")
step = 0
def say(first, *rest):
    """Number the steps as they are printed, so a skipped step leaves no gap."""
    global step
    step += 1
    print(f"  {step}. {first}")
    for line in rest:
        print(f"     {line}")
# Two different instructions, never one with a hole in it. A notebook that has not
# run is named; a file that is missing although its notebook ran is a different
# problem and sends the reader to a different place.
if NOT_RUN:
    say("Run the notebooks that have not: " + ", ".join(NOT_RUN) + ",",
        "then run this notebook again. It overwrites, so re-running is free.")
elif missing_core:
    say("Every notebook has run, but files something needs are not in the",
        "folder -- they are listed above, with the notebook that owes each one.",
        "Re-run those notebooks, then run this one again.")
say(f"Unzip {BUNDLE} and put the contents of its {FOR_RMD}/ folder into",
    "Analysis/NLP/outputs/, then knit 02_nlp_integration.Rmd. That is the",
    "entire integration step: the document finds the six files by name.")
say("Any section of the knitted document that prints an AWAITING COLAB",
    "box is a file that did not arrive, not an analysis that failed.")
if deposit_gaps or stale_human:
    say("Before this goes to OSF, close the two lines above: the deposit is",
        "missing a result table, or the folder holds output from the human arm",
        "in a run that reports itself as the registered pipeline.")
say("Before posting anything to OSF: clear the verbatim participant text",
    "listed above, and post the dated amendments in MEASUREMENT_DESIGN.md",
    "section 0 for every deviation this run relied on.")
print("\n  Nothing in this chain is confirmatory. E3 to E7 are one exploratory")
print("  family (prereg 2.4), and the gate decides standing inside it, not")
print("  evidential status.")

---
## Done

- **For the R Markdown:** `for_rmd/` holds the six files `02_nlp_integration.Rmd`
  reads, under the names it reads them by -- `ensemble_facet_scores.csv`,
  `ensemble_agreement.csv`, `embedding_facet_similarity.csv`,
  `facet_dictionary_scores.csv`, `gold_codes.csv` (only when the human arm ran)
  and `gold_split_assignment.csv`. Drop them into `Analysis/NLP/outputs/` and
  re-knit. The folder is emptied at the start of every run, so what is in it is
  what this run wrote and nothing else.
- **For OSF:** `retribution_nlp_bundle.zip` is the whole chain, and
  `osf_bundle_manifest.json` is the written list of what the deposit has to
  contain, so a gap in it is visible rather than silent. Five tables are named as
  the ones the deposit cannot go up without. De-identify the verbatim participant
  text first.
- **The gate:** read back from `gate_results.csv`, never recomputed here. Clauses B
  (models agree at ICC >= .60) and C (each facet correlates r >= .30 with its
  matching survey scale) are the registered gate of prereg 6.7. Clause A,
  agreement with human coders, is this project's addition and runs only when
  `HUMAN_GOLD` is True; where it did not run the table says "not run" and never
  leaves a blank that could be misread as a failure.
- **Gate-backed is not confirmatory.** Every E3 to E7 test is exploratory (prereg
  2.4). The gate decides whether a facet is primary or descriptive inside that
  one exploratory family, and nothing more.

### If a section of the knitted document is empty

The R Markdown replaces any section whose input it cannot find with a boxed
AWAITING COLAB OUTPUTS notice. That notice means a file did not arrive under the
name it was looked for -- it does not mean the analysis was run and returned
nothing. The state report at the top of this notebook says which notebook owes
which file.

### If this notebook stops

One thing stops it: a file on its way to R that carries the same
`response_id x item_type` more than once. That is the one defect nothing
downstream would catch, since the R Markdown averages the repeat and prints the
sample size it always did. The message names the file and the notebook that
writes it. Everything else -- a missing input, an unreadable table, a chain that
is half finished -- is reported and the run carries on.

### Before the deposit goes up

The deviations this design takes from the registration -- the human gold standard
and gate clause A, the proportionality demotion, the 5% word-list coverage rule,
the tightened clause C, the CI-bound gating, and the E4 split-call scoring -- each
go up as a dated amendment. The registration is approved and out of embargo, so
an amendment posted before the ensemble scored the corpus is what keeps any of
these choices from being read as post-hoc. The full table is
`MEASUREMENT_DESIGN.md` section 0.